In [6]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_febros_fractions_df,
    wade_febros_scaler,
    wade_febros_pca,
    wade_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-02-14 00:00:00",
    end_date="2023-02-20 00:00:00",
    endmember_ids=["RI23-5018", "RI23-5000", "RI23-5005"],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-5000", "RI23-5005"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-14 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-20 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.7 
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1009 2023-02-15 15:00:00
1  RI23-1010 2023-02-15 19:00:00
2  RI23-1011 2023-02-15 23:00:00
3  RI23-1025 2023-02-15 12:00:00
4  RI23-1012 2023-02-16 03:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_70sig,Snowmelt lysimeter_Uncertainty_70sig,Soil water lysimeter_Uncertainty_70sig
0,RI23-1009,2023-02-15 15:00:00,Wade,0.499577,0.482952,1.747075e-02,1.0,0.134753,0.134753,1.011541e-12
1,RI23-1010,2023-02-15 19:00:00,Wade,0.450879,0.431147,1.179747e-01,1.0,0.117787,0.285699,2.916669e-01
2,RI23-1011,2023-02-15 23:00:00,Wade,0.414333,0.585667,1.131854e-15,1.0,0.112060,0.112060,2.968197e-12
3,RI23-1025,2023-02-15 12:00:00,Wade,0.429392,0.465132,1.054759e-01,1.0,0.117554,0.292605,2.976261e-01
4,RI23-1012,2023-02-16 03:00:00,Wade,0.245741,0.462826,2.914328e-01,1.0,0.092950,0.277451,2.650986e-01
5,RI23-1014,2023-02-16 11:00:00,Wade,0.249144,0.557428,1.934282e-01,1.0,0.085727,0.266095,2.642566e-01
6,RI23-1028,2023-02-16 14:00:00,Wade,0.237028,0.541212,2.217600e-01,1.0,0.084021,0.254730,2.558992e-01
7,RI23-1029,2023-02-16 20:00:00,Wade,0.313152,0.602808,8.403991e-02,1.0,0.084467,0.263121,2.637632e-01
8,RI23-1030,2023-02-17 02:00:00,Wade,0.269277,0.489367,2.413563e-01,1.0,0.093446,0.093446,3.533966e-13
9,RI23-1031,2023-02-17 08:00:00,Wade,0.284139,0.500626,2.152350e-01,1.0,0.090072,0.268654,2.692253e-01


In [5]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_martherm_fractions_df,
    wade_martherm_scaler,
    wade_martherm_pca,
    wade_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   "RI23-5018", #"RI23-5006", # Homeowner well groundwater March 2023
                   #"RI23-1034", # Pre-event baseflow, labeled as GW in Wade index
                   "RI23-5005", # Snowmelt lysimeter 02/15/2023
                   #"RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-5009", # Soil water lysimeter march
                   ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-5005", "RI23-5009"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_martherm_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.7
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1039 2023-03-22 18:00:00
1  RI23-1040 2023-03-23 00:00:00
2  RI23-1041 2023-03-23 06:00:00
3  RI23-1055 2023-03-23 12:00:00
4  RI23-1056 2023-03-23 18:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_70sig,Snowmelt lysimeter_Uncertainty_70sig,Soil water lysimeter_Uncertainty_70sig
0,RI23-1039,2023-03-22 18:00:00,Wade,0.554301,0.445699,2.078197e-14,1.0,0.134375,0.134375,9.296233e-13
1,RI23-1040,2023-03-23 00:00:00,Wade,0.492368,0.370275,1.373570e-01,1.0,0.129236,0.222604,1.725792e-01
2,RI23-1041,2023-03-23 06:00:00,Wade,0.504608,0.495392,9.470066e-15,1.0,0.128645,0.128645,7.046021e-13
3,RI23-1055,2023-03-23 12:00:00,Wade,0.493162,0.506838,0.000000e+00,1.0,0.125419,0.125419,1.819112e-13
4,RI23-1056,2023-03-23 18:00:00,Wade,0.324627,0.476912,1.984611e-01,1.0,0.099558,0.194837,1.606318e-01
5,RI23-1057,2023-03-24 00:00:00,Wade,0.245196,0.481885,2.729187e-01,1.0,0.085013,0.183292,1.565600e-01
6,RI23-1058,2023-03-24 06:00:00,Wade,0.224620,0.432609,3.427703e-01,1.0,0.083384,0.183938,1.583107e-01
7,RI23-1059,2023-03-24 12:00:00,Wade,0.437939,0.424008,1.380526e-01,1.0,0.118015,0.214915,1.709736e-01
8,RI23-1060,2023-03-24 17:30:00,Wade,0.337822,0.533364,1.288134e-01,1.0,0.098409,0.189476,1.560675e-01


In [4]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

#wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']
wade_tracers = ['Ca_mg_L', 'Mg_mg_L', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_fmelt_fractions_df,
    wade_fmelt_scaler,
    wade_fmelt_pca,
    wade_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-15 00:00:00",
    endmember_ids=[
                   "RI23-5018", #"RI23-5006", # Homeowner well groundwater March 2023
                   #"RI22-0860", "RI22-0859", # Homeowner well groundwater 2022
                   #"RI23-1034", # Pre-event baseflow, labeled as GW in Wade index
                   #"RI25-1111", "RI25-1126", "RI25-1183", "RI25-1201", "RI25-1290", # 2025 baseflow samples 
                   #"RI23-5005", # Snowmelt lysimeter 02/15/2023
                   #"RI23-1063", # Snowmelt lysimeter 3/28
                   "RI23-1098", # Snowmelt lysimeter 04/11/2023
                   #"RI23-5009", # Soil water lysimeter sample
                   "RI23-5011", # Soil water lysimeter wet 4/12
                   ],  
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5018", "RI23-1098", "RI23-5011"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-15 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_fmelt_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
    confidence_level = 0.7
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1064 2023-03-31 08:00:00
1  RI23-1065 2023-03-31 14:00:00
2  RI23-1066 2023-03-31 20:00:00
3  RI23-1067 2023-04-01 02:00:00
4  RI23-1068 2023-04-01 08:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_70sig,Snowmelt lysimeter_Uncertainty_70sig,Soil water lysimeter_Uncertainty_70sig
0,RI23-1064,2023-03-31 08:00:00,Wade,0.500580,0.000000e+00,0.499420,1.0,0.221773,0.479350,6.791582e-01
1,RI23-1065,2023-03-31 14:00:00,Wade,0.374702,8.000588e-16,0.625298,1.0,0.189009,0.425530,5.949642e-01
2,RI23-1066,2023-03-31 20:00:00,Wade,0.457860,0.000000e+00,0.542140,1.0,0.204813,0.450343,6.350603e-01
3,RI23-1067,2023-04-01 02:00:00,Wade,0.455074,2.768781e-15,0.544926,1.0,0.213020,0.468757,6.602446e-01
4,RI23-1068,2023-04-01 08:00:00,Wade,0.481835,1.067804e-16,0.518165,1.0,0.219111,0.470274,6.686802e-01
5,RI23-1070,2023-04-01 20:00:00,Wade,0.247095,1.286539e-01,0.624251,1.0,0.166856,0.387129,5.350503e-01
6,RI23-1093,2023-04-10 06:00:00,Wade,0.272603,2.862335e-01,0.441164,1.0,0.157476,0.375053,5.137567e-01
7,RI23-1094,2023-04-10 12:00:00,Wade,0.218866,3.551210e-01,0.426013,1.0,0.158190,0.370746,5.111131e-01
8,RI23-1095,2023-04-10 18:00:00,Wade,0.185571,2.968508e-01,0.517578,1.0,0.145587,0.354645,4.821649e-01
9,RI23-1096,2023-04-11 00:00:00,Wade,0.181204,3.623920e-01,0.456404,1.0,0.141867,0.355346,4.797590e-01
